# Corrected Association Rules & Hybrid Recommendation System
## Amazon All Beauty reviews, 2015+ — cluster-integrated final version

Association rules are framed as supplementary co-engagement evidence supporting the recommendation analysis. The recommender is evaluated temporally with rating and top-10 ranking metrics, and results are connected to the final four product clusters.

In [1]:
from pathlib import Path
import ast
import re
import warnings
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.sentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
from IPython.display import display

DATA_PATH = Path("amazon_beauty_full_2015plus.csv.gz")
CLUSTER_PATH = Path("product_clusters_k4_2015plus_v3.csv")
OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {DATA_PATH}")

## 1. Load and validate the canonical 2015+ data

In [3]:
usecols = [
    "user_id", "asin", "rating", "text", "review_date",
    "review_year", "product_title",
]
df = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    parse_dates=["review_date"],
    low_memory=False,
)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = (
    df.dropna(subset=["user_id", "asin", "rating", "review_date"])
    .drop_duplicates(["user_id", "asin"], keep="first")
    .copy()
)
date_check = df["review_date"].min() >= pd.Timestamp("2015-01-01")
year_check = df["review_year"].min() >= 2015
print(f"[{'PASS' if date_check else 'WARNING'}] Earliest review date is 2015-01-01 or later.")
print(f"[{'PASS' if year_check else 'WARNING'}] Earliest review year is 2015 or later.")

print(f"Reviews: {len(df):,}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Products: {df['asin'].nunique():,}")
print(f"Date range: {df['review_date'].min()} to {df['review_date'].max()}")

In [4]:
cluster_lookup = pd.read_csv(
    CLUSTER_PATH, usecols=["asin", "cluster", "cluster_name"]
).drop_duplicates("asin")
cluster_name_map = cluster_lookup.set_index("asin")["cluster_name"]
cluster_id_map = cluster_lookup.set_index("asin")["cluster"]
matched_reviews = df["asin"].isin(cluster_name_map.index).sum()
print(f"Cluster lookup: {len(cluster_lookup):,} products")
print(f"Reviews with a cluster assignment: {matched_reviews:,} ({matched_reviews/len(df):.1%})")
print("Unmatched products remain eligible for the general recommender; cluster-level results are reported only where assignments exist.")

## 2. Association rules from review co-occurrence

The transaction population is every user who reviewed at least two distinct
products. Items are restricted to the 1,000 most frequent products for
tractability, but users whose restricted basket becomes empty remain in the
matrix. Consequently, support uses all multi-product reviewers as its
denominator rather than only the users left with two popular products.

In [6]:
baskets = df.groupby("user_id")["asin"].agg(lambda values: sorted(set(values)))
multi_baskets = baskets[baskets.map(len).ge(2)]
n_all_users = len(baskets)
n_multi_users = len(multi_baskets)

frequencies = Counter()
for basket in multi_baskets:
    frequencies.update(basket)
top_items = set(
    pd.Series(frequencies).sort_values(ascending=False).head(1000).index
)
restricted_baskets = [
    [item for item in basket if item in top_items]
    for basket in multi_baskets
]

min_pair_count = 10
min_support = min_pair_count / n_multi_users

singleton_counts = Counter()
pair_counts = Counter()
for basket in restricted_baskets:
    unique_items = sorted(set(basket))
    singleton_counts.update(unique_items)
    pair_counts.update(combinations(unique_items, 2))

directional_rows = []
for (item_a, item_b), pair_count in pair_counts.items():
    if pair_count < min_pair_count:
        continue
    pair_support = pair_count / n_multi_users
    for antecedent, consequent in [(item_a, item_b), (item_b, item_a)]:
        antecedent_support = singleton_counts[antecedent] / n_multi_users
        consequent_support = singleton_counts[consequent] / n_multi_users
        confidence = pair_support / antecedent_support
        lift = confidence / consequent_support
        if confidence < .20 or lift < 1.2:
            continue
        leverage = pair_support - antecedent_support * consequent_support
        conviction = (
            (1 - consequent_support) / (1 - confidence)
            if confidence < 1 else np.inf
        )
        directional_rows.append({
            "antecedent_asin": antecedent,
            "consequent_asin": consequent,
            "antecedent support": antecedent_support,
            "consequent support": consequent_support,
            "support": pair_support,
            "confidence": confidence,
            "lift": lift,
            "leverage": leverage,
            "conviction": conviction,
        })
directional_rules = pd.DataFrame(directional_rows)

print(f"All users: {n_all_users:,}")
print(f"Multi-product reviewers: {n_multi_users:,}")
print(
    f"Restricted item universe: {len(top_items):,} products across "
    f"{n_multi_users:,} multi-product users"
)
print(f"Minimum pair count: {min_pair_count}")
print(f"Directional rules before reciprocal consolidation: {len(directional_rules)}")

In [7]:
asin_to_title = (
    df.drop_duplicates("asin").set_index("asin")["product_title"].to_dict()
)

rules = directional_rules.copy()
rules["pair_key"] = rules.apply(
    lambda row: " | ".join(sorted([
        row["antecedent_asin"], row["consequent_asin"]
    ])),
    axis=1,
)
rules["cooccurrence_users"] = (
    rules["support"] * n_multi_users
).round().astype(int)
rules["support_all_users"] = (
    rules["cooccurrence_users"] / n_all_users
)
rules["antecedent_title"] = rules["antecedent_asin"].map(asin_to_title)
rules["consequent_title"] = rules["consequent_asin"].map(asin_to_title)

# Retain the stronger-confidence direction for each reciprocal pair.
rules_report = (
    rules.sort_values(
        ["pair_key", "confidence", "lift"],
        ascending=[True, False, False],
    )
    .drop_duplicates("pair_key", keep="first")
    .sort_values(["lift", "cooccurrence_users"], ascending=False)
    .reset_index(drop=True)
)

rule_columns = [
    "antecedent_asin", "antecedent_title",
    "consequent_asin", "consequent_title",
    "cooccurrence_users", "support", "support_all_users",
    "confidence", "lift", "leverage", "conviction",
]
rules_report[rule_columns].to_csv(
    OUTPUT_DIR / "association_rules_corrected_2015plus.csv",
    index=False,
)
print(f"Unique reported pairs: {len(rules_report)}")
rules_report[rule_columns].head(10)

In [8]:
rules_report["antecedent_cluster"] = rules_report["antecedent_asin"].map(cluster_name_map)
rules_report["consequent_cluster"] = rules_report["consequent_asin"].map(cluster_name_map)
rules_report["same_cluster"] = (
    rules_report["antecedent_cluster"].notna()
    & rules_report["antecedent_cluster"].eq(rules_report["consequent_cluster"])
)
cluster_rule_columns = rule_columns + [
    "antecedent_cluster", "consequent_cluster", "same_cluster"
]
rules_report[cluster_rule_columns].to_csv(
    OUTPUT_DIR / "association_rules_cluster_enriched_2015plus.csv", index=False
)
print(f"Rules with both products clustered: {rules_report[['antecedent_cluster','consequent_cluster']].notna().all(axis=1).sum():,}")
display(rules_report[cluster_rule_columns].head(10))

In [9]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    rules_report["support_all_users"],
    rules_report["confidence"],
    c=rules_report["lift"],
    s=45 + 4 * rules_report["cooccurrence_users"],
    cmap="viridis",
    alpha=.8,
    edgecolor="white",
)
fig.colorbar(scatter, ax=ax, label="Lift")
ax.set(
    title="Review co-occurrence rules (unique product pairs)",
    xlabel="Support among all users",
    ylabel="Confidence",
)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "association_rules_corrected_2015plus.png",
    dpi=180, bbox_inches="tight",
)
plt.show()

## 3. Build an iterative interaction core

The original 3-user/10-item thresholds collapse to zero interactions when
enforced repeatedly. A documented 3-user/5-item k-core is therefore used.
Filtering repeats until every retained user has at least three interactions
and every retained product has at least five.

In [11]:
MIN_USER_INTERACTIONS = 3
MIN_ITEM_INTERACTIONS = 5

core = df.copy()
previous_size = -1
iterations = 0
while len(core) != previous_size:
    previous_size = len(core)
    user_counts = core["user_id"].value_counts()
    item_counts = core["asin"].value_counts()
    core = core[
        core["user_id"].isin(
            user_counts[user_counts.ge(MIN_USER_INTERACTIONS)].index
        )
        & core["asin"].isin(
            item_counts[item_counts.ge(MIN_ITEM_INTERACTIONS)].index
        )
    ].copy()
    iterations += 1

user_core_check = core.groupby("user_id").size().min() >= MIN_USER_INTERACTIONS
item_core_check = core.groupby("asin").size().min() >= MIN_ITEM_INTERACTIONS
print(f"[{'PASS' if user_core_check else 'WARNING'}] Every core user has at least {MIN_USER_INTERACTIONS} interactions.")
print(f"[{'PASS' if item_core_check else 'WARNING'}] Every core item has at least {MIN_ITEM_INTERACTIONS} interactions.")
print(
    f"Iterative core: {len(core):,} interactions, "
    f"{core['user_id'].nunique():,} users, "
    f"{core['asin'].nunique():,} products ({iterations} passes)"
)

## 4. Temporal train/validation/test split

For each user, the most recent interaction is test, the second-most recent
is validation, and all earlier interactions are training. Evaluation is
restricted to warm-start products present in the relevant training data,
with coverage reported rather than silently treating unknown products as
known.

In [13]:
core = core.sort_values(["user_id", "review_date", "asin"]).copy()
core["position"] = core.groupby("user_id").cumcount()
core["user_n"] = core.groupby("user_id")["asin"].transform("size")

train = core[core["position"].lt(core["user_n"] - 2)].copy()
validation = core[core["position"].eq(core["user_n"] - 2)].copy()
test = core[core["position"].eq(core["user_n"] - 1)].copy()

val_warm = validation[
    validation["asin"].isin(train["asin"].unique())
].copy()
train_plus_val = pd.concat([train, validation], ignore_index=True)
test_warm = test[
    test["asin"].isin(train_plus_val["asin"].unique())
].copy()

print(f"Train/validation/test rows: {len(train):,}/{len(validation):,}/{len(test):,}")
print(f"Validation warm-item coverage: {len(val_warm) / len(validation):.1%}")
print(f"Test warm-item coverage: {len(test_warm) / len(test):.1%}")

## 5. Rating-only matrix factorization on observed ratings

The model uses stochastic gradient descent with user and item biases and
latent factors. Loss is evaluated only on observed interactions; missing
entries are never treated as zero ratings.

In [15]:
class ExplicitMF:
    def __init__(
        self, n_factors=20, learning_rate=.01, regularization=.05,
        epochs=40, random_state=42,
    ):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.epochs = epochs
        self.random_state = random_state

    def fit(self, frame):
        self.user_ids = frame["user_id"].unique()
        self.item_ids = frame["asin"].unique()
        self.u2i = {value: i for i, value in enumerate(self.user_ids)}
        self.p2i = {value: i for i, value in enumerate(self.item_ids)}
        self.global_mean = frame["rating"].mean()
        random = np.random.default_rng(self.random_state)
        self.user_factors = random.normal(
            0, .1, (len(self.user_ids), self.n_factors)
        )
        self.item_factors = random.normal(
            0, .1, (len(self.item_ids), self.n_factors)
        )
        self.user_bias = np.zeros(len(self.user_ids))
        self.item_bias = np.zeros(len(self.item_ids))
        triples = np.column_stack([
            frame["user_id"].map(self.u2i).to_numpy(),
            frame["asin"].map(self.p2i).to_numpy(),
            frame["rating"].to_numpy(),
        ])

        for _ in range(self.epochs):
            for index in random.permutation(len(triples)):
                user, item, rating = triples[index]
                user, item = int(user), int(item)
                prediction = (
                    self.global_mean
                    + self.user_bias[user]
                    + self.item_bias[item]
                    + self.user_factors[user] @ self.item_factors[item]
                )
                error = rating - prediction
                self.user_bias[user] += self.learning_rate * (
                    error - self.regularization * self.user_bias[user]
                )
                self.item_bias[item] += self.learning_rate * (
                    error - self.regularization * self.item_bias[item]
                )
                old_user = self.user_factors[user].copy()
                self.user_factors[user] += self.learning_rate * (
                    error * self.item_factors[item]
                    - self.regularization * self.user_factors[user]
                )
                self.item_factors[item] += self.learning_rate * (
                    error * old_user
                    - self.regularization * self.item_factors[item]
                )
        return self

    def predict_pairs(self, frame):
        predictions = []
        for user_id, asin in zip(frame["user_id"], frame["asin"]):
            if user_id not in self.u2i or asin not in self.p2i:
                predictions.append(self.global_mean)
                continue
            user, item = self.u2i[user_id], self.p2i[asin]
            predictions.append(
                self.global_mean
                + self.user_bias[user]
                + self.item_bias[item]
                + self.user_factors[user] @ self.item_factors[item]
            )
        return np.clip(np.asarray(predictions), 1, 5)

    def score_all_items(self, user_id):
        user = self.u2i[user_id]
        scores = (
            self.global_mean + self.user_bias[user] + self.item_bias
            + self.item_factors @ self.user_factors[user]
        )
        return np.clip(scores, 1, 5)

def rmse(actual, predicted):
    return float(np.sqrt(np.mean((np.asarray(actual) - predicted) ** 2)))

def mae(actual, predicted):
    return float(np.mean(np.abs(np.asarray(actual) - predicted)))

## 6. Tune matrix-factorization complexity on validation data

In [17]:
mf_trials = []
for factors in [5, 10, 20, 40]:
    candidate = ExplicitMF(
        n_factors=factors, epochs=50, random_state=RANDOM_STATE
    ).fit(train)
    predictions = candidate.predict_pairs(val_warm)
    mf_trials.append({
        "n_factors": factors,
        "validation_rmse": rmse(val_warm["rating"], predictions),
        "validation_mae": mae(val_warm["rating"], predictions),
    })

mf_validation = pd.DataFrame(mf_trials)
BEST_FACTORS = int(
    mf_validation.sort_values("validation_rmse").iloc[0]["n_factors"]
)
print(f"Selected latent factors: {BEST_FACTORS}")
mf_validation

## 7. Create training-only product sentiment and tune the hybrid weight

Sentiment is computed only from reviews available in the current training
split. No validation or test review text contributes to its own product
feature.

In [19]:
vader = SentimentIntensityAnalyzer()

def score_text(value):
    if pd.isna(value) or not str(value).strip():
        return 0.0
    return vader.polarity_scores(str(value))["compound"]

train = train.copy()
train["sentiment_compound"] = train["text"].map(score_text)
train_item_sentiment = train.groupby("asin")["sentiment_compound"].mean()
train_sentiment_global = train_item_sentiment.mean()

def scaled_sentiment(frame, series, global_value):
    compound = frame["asin"].map(series).fillna(global_value)
    return 3 + 2 * compound

validation_sentiment = scaled_sentiment(
    val_warm, train_item_sentiment, train_sentiment_global
).clip(1, 5)

validation_model = ExplicitMF(
    n_factors=BEST_FACTORS, epochs=50, random_state=RANDOM_STATE
).fit(train)
validation_cf = validation_model.predict_pairs(val_warm)

blend_trials = []
for alpha in [1.0, .9, .8, .7, .6, .5, .3]:
    prediction = np.clip(
        alpha * validation_cf + (1 - alpha) * validation_sentiment,
        1, 5,
    )
    blend_trials.append({
        "alpha_cf_weight": alpha,
        "validation_rmse": rmse(val_warm["rating"], prediction),
        "validation_mae": mae(val_warm["rating"], prediction),
    })
blend_validation = pd.DataFrame(blend_trials)
BEST_ALPHA = float(
    blend_validation.sort_values("validation_rmse").iloc[0][
        "alpha_cf_weight"
    ]
)
print(f"Selected CF blend weight: {BEST_ALPHA:.1f}")
blend_validation

## 8. Refit on train + validation and evaluate the untouched test set

In [21]:
train_plus_val = train_plus_val.copy()
train_plus_val["sentiment_compound"] = train_plus_val["text"].map(score_text)
final_item_sentiment = (
    train_plus_val.groupby("asin")["sentiment_compound"].mean()
)
final_sentiment_global = final_item_sentiment.mean()

final_model = ExplicitMF(
    n_factors=BEST_FACTORS, epochs=60, random_state=RANDOM_STATE
).fit(train_plus_val)
cf_test = final_model.predict_pairs(test_warm)
sentiment_test = scaled_sentiment(
    test_warm, final_item_sentiment, final_sentiment_global
).clip(1, 5).to_numpy()
hybrid_test = np.clip(
    BEST_ALPHA * cf_test + (1 - BEST_ALPHA) * sentiment_test,
    1, 5,
)

global_prediction = np.full(
    len(test_warm), train_plus_val["rating"].mean()
)
item_means = train_plus_val.groupby("asin")["rating"].mean()
item_mean_prediction = (
    test_warm["asin"].map(item_means)
    .fillna(train_plus_val["rating"].mean()).to_numpy()
)

comparison = pd.DataFrame([
    {
        "model": "Global mean",
        "RMSE": rmse(test_warm["rating"], global_prediction),
        "MAE": mae(test_warm["rating"], global_prediction),
    },
    {
        "model": "Item mean",
        "RMSE": rmse(test_warm["rating"], item_mean_prediction),
        "MAE": mae(test_warm["rating"], item_mean_prediction),
    },
    {
        "model": "Rating-only MF",
        "RMSE": rmse(test_warm["rating"], cf_test),
        "MAE": mae(test_warm["rating"], cf_test),
    },
    {
        "model": "Hybrid MF + sentiment",
        "RMSE": rmse(test_warm["rating"], hybrid_test),
        "MAE": mae(test_warm["rating"], hybrid_test),
    },
])
comparison["test_rows"] = len(test_warm)
comparison["test_coverage"] = len(test_warm) / len(test)
comparison["selected_factors"] = BEST_FACTORS
comparison["selected_alpha"] = BEST_ALPHA
comparison.to_csv(
    OUTPUT_DIR / "recommendation_comparison_corrected_2015plus.csv",
    index=False,
)
comparison

## 9. Top-10 ranking evaluation

Each eligible user's positive held-out interaction (`rating >= 4`) is ranked
against all warm-start products not already seen in train + validation.
With one relevant test item per user, Recall@10 equals HitRate@10 and
Precision@10 equals hits divided by ten.

In [23]:
seen_by_user = train_plus_val.groupby("user_id")["asin"].agg(set).to_dict()
positive_test = test_warm[test_warm["rating"].ge(4)].copy()

ranking_rows = []
example_rows = []
item_titles = (
    df.drop_duplicates("asin").set_index("asin")["product_title"].to_dict()
)
final_sentiment_scaled = (
    3 + 2 * final_item_sentiment.reindex(final_model.item_ids)
    .fillna(final_sentiment_global)
).clip(1, 5).to_numpy()

for row in positive_test.itertuples(index=False):
    user_id, target_item = row.user_id, row.asin
    if user_id not in final_model.u2i or target_item not in final_model.p2i:
        continue
    cf_scores = final_model.score_all_items(user_id)
    hybrid_scores = np.clip(
        BEST_ALPHA * cf_scores
        + (1 - BEST_ALPHA) * final_sentiment_scaled,
        1, 5,
    )
    excluded = seen_by_user.get(user_id, set())
    allowed = np.array([
        item not in excluded for item in final_model.item_ids
    ])
    target_index = final_model.p2i[target_item]
    allowed[target_index] = True

    for model_name, scores in [
        ("Rating-only MF", cf_scores),
        ("Hybrid MF + sentiment", hybrid_scores),
    ]:
        candidates = np.where(allowed)[0]
        ranked = candidates[np.argsort(scores[candidates])[::-1]]
        rank_positions = np.where(ranked == target_index)[0]
        rank = int(rank_positions[0] + 1)
        hit10 = int(rank <= 10)
        ranking_rows.append({
            "model": model_name,
            "user_id": user_id,
            "target_asin": target_item,
            "target_rating": row.rating,
            "rank": rank,
            "hit_at_10": hit10,
            "ndcg_at_10": 1 / np.log2(rank + 1) if hit10 else 0,
            "reciprocal_rank_at_10": 1 / rank if hit10 else 0,
        })

    if len(example_rows) < 50:
        top10 = np.where(allowed)[0][
            np.argsort(hybrid_scores[allowed])[::-1][:10]
        ]
        for recommendation_rank, item_index in enumerate(top10, 1):
            item = final_model.item_ids[item_index]
            example_rows.append({
                "user_id": user_id,
                "rank": recommendation_rank,
                "recommended_asin": item,
                "product_title": item_titles.get(item, ""),
                "hybrid_score": hybrid_scores[item_index],
            })

ranking_detail = pd.DataFrame(ranking_rows)
ranking_summary = ranking_detail.groupby("model").agg(
    evaluated_users=("user_id", "size"),
    hit_rate_at_10=("hit_at_10", "mean"),
    recall_at_10=("hit_at_10", "mean"),
    ndcg_at_10=("ndcg_at_10", "mean"),
    mrr_at_10=("reciprocal_rank_at_10", "mean"),
).reset_index()
ranking_summary["precision_at_10"] = (
    ranking_summary["hit_rate_at_10"] / 10
)
ranking_summary.to_csv(
    OUTPUT_DIR / "recommendation_ranking_metrics_corrected_2015plus.csv",
    index=False,
)
pd.DataFrame(example_rows).to_csv(
    OUTPUT_DIR / "recommendation_examples_corrected_2015plus.csv",
    index=False,
)
ranking_summary

In [24]:
ranking_detail["target_cluster"] = ranking_detail["target_asin"].map(cluster_name_map)
ranking_by_cluster = (
    ranking_detail.dropna(subset=["target_cluster"])
    .groupby(["model", "target_cluster"])
    .agg(
        evaluated_cases=("user_id", "size"),
        hit_rate_at_10=("hit_at_10", "mean"),
        ndcg_at_10=("ndcg_at_10", "mean"),
        mrr_at_10=("reciprocal_rank_at_10", "mean"),
    ).reset_index()
)
ranking_by_cluster.to_csv(
    OUTPUT_DIR / "recommendation_metrics_by_cluster_2015plus.csv", index=False
)
examples = pd.DataFrame(example_rows)
examples["recommended_cluster"] = examples["recommended_asin"].map(cluster_name_map)
examples.to_csv(
    OUTPUT_DIR / "recommendation_examples_cluster_labeled_2015plus.csv", index=False
)
display(ranking_by_cluster)

if not ranking_by_cluster.empty:
    plot_data = ranking_by_cluster.pivot(
        index="target_cluster", columns="model", values="hit_rate_at_10"
    )
    ax = plot_data.plot.bar(figsize=(10, 5), color=["#4E79A7", "#F28E2B"])
    ax.set(title="Top-10 recommendation performance by target cluster", xlabel="", ylabel="HitRate@10")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "recommendation_performance_by_cluster_2015plus.png", dpi=180, bbox_inches="tight")
    plt.show()

## 10. Final figures

In [26]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(
    comparison["model"], comparison["RMSE"],
    color=["#BAB0AC", "#9C755F", "#4E79A7", "#F28E2B"],
)
axes[0].set(title="Untouched temporal test RMSE", ylabel="RMSE")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_ylim(
    comparison["RMSE"].min() - .03,
    comparison["RMSE"].max() + .03,
)

metric_plot = ranking_summary.set_index("model")[
    ["hit_rate_at_10", "ndcg_at_10"]
]
metric_plot.plot.bar(ax=axes[1], color=["#59A14F", "#E15759"])
axes[1].set(
    title="Top-10 ranking performance",
    ylabel="Score", xlabel="",
)
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(["HitRate@10", "NDCG@10"])
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "recommendation_evaluation_corrected_2015plus.png",
    dpi=180, bbox_inches="tight",
)
plt.show()

## 11. Quality control and report-ready interpretation

In [28]:
latest_training_date = train_plus_val.groupby("user_id")[
    "review_date"
].max()
quality_checks = {
    "Reviews begin on or after 2015-01-01": df["review_date"].min() >= pd.Timestamp("2015-01-01"),
    f"Every core user has at least {MIN_USER_INTERACTIONS} interactions": core.groupby("user_id").size().min() >= MIN_USER_INTERACTIONS,
    f"Every core item has at least {MIN_ITEM_INTERACTIONS} interactions": core.groupby("asin").size().min() >= MIN_ITEM_INTERACTIONS,
    "Test interactions occur after each user's latest training interaction": (
        test_warm["review_date"]
        >= test_warm["user_id"].map(latest_training_date)
    ).all(),
    "All RMSE values are available": comparison["RMSE"].notna().all(),
    "All HitRate@10 values are between 0 and 1": ranking_summary["hit_rate_at_10"].between(0, 1).all(),
    "Reported association-rule pair keys are unique": rules_report["pair_key"].is_unique,
}
for check_name, passed in quality_checks.items():
    print(f"[{'PASS' if passed else 'WARNING'}] {check_name}")

hybrid_rmse = comparison.loc[
    comparison["model"].eq("Hybrid MF + sentiment"), "RMSE"
].iloc[0]
rating_rmse = comparison.loc[
    comparison["model"].eq("Rating-only MF"), "RMSE"
].iloc[0]
rmse_change = (rating_rmse - hybrid_rmse) / rating_rmse
ranking_lookup = ranking_summary.set_index("model")
hybrid_hit = ranking_lookup.loc[
    "Hybrid MF + sentiment", "hit_rate_at_10"
]
rating_hit = ranking_lookup.loc[
    "Rating-only MF", "hit_rate_at_10"
]
hybrid_ndcg = ranking_lookup.loc[
    "Hybrid MF + sentiment", "ndcg_at_10"
]
rating_ndcg = ranking_lookup.loc[
    "Rating-only MF", "ndcg_at_10"
]

if all(quality_checks.values()):
    print("All quality-control checks passed.")
else:
    print("One or more quality-control checks raised a warning; review the messages above.")
print(f"Hybrid RMSE change versus rating-only MF: {rmse_change:.2%}")
print(
    f"HitRate@10 - hybrid: {hybrid_hit:.2%}; "
    f"rating-only: {rating_hit:.2%}"
)
print(
    f"NDCG@10 - hybrid: {hybrid_ndcg:.4f}; "
    f"rating-only: {rating_ndcg:.4f}"
)
print(
    "Interpret improvements cautiously: this is one temporal split over a "
    "small iterative core. The result applies to warm-start products and "
    "moderately active users, not the full Amazon Beauty long tail."
)

## Report-ready conclusion

The association rules identify repeated review co-engagement patterns rather than purchases and therefore must not be described as market baskets or causal product affinities. The recommendation system uses a temporal train/validation/test design, compares against simple baselines, and reports both rating-prediction and top-10 ranking metrics. Cluster-enriched outputs show which segments occur in rules and whether recommendation performance differs by target segment, directly connecting this supplementary module to the project's segmentation results. Conclusions apply mainly to warm-start products and active users; cluster-level conclusions apply only to products receiving a final cluster assignment.